# Tracking videos: Reference rotating square design (3dp pla + blue shims) for multiloading source cloaking

## Imports

In [ ]:
import jax
import matplotlib.pyplot as plt
from typing import Union
from mechanicalmetamaterialcloaks.plotting import generate_animation
from mechanicalmetamaterialcloaks.geometry import current_coordinates
from mechanicalmetamaterialcloaks.utils import save_data, load_data, SolutionData
from problems.quads_dynamic_load_shielding_source_multi_loading import OptimizationProblem
import cv2

from pathlib import Path
import pandas as pd
from scripts.tracking.tracking_gray_xcorr import mark_reference_frame
from scripts.tracking.tracking_gray_xcorr import tracking_better as tracking
from scripts.tracking.utils import smooth_fields_SG
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)  # enable float64 type

plt.style.use(["science", "grid"])
%matplotlib widget


No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


## Paths

In [ ]:
# Where to load/save data
data_folder = Path("../../data/quads_shielding_multi_loading_source_3dp_pla_shims_4_loads_reference")
# Where to save plots and animations
out_folder = Path("../../out/quads_shielding_multi_loading_source_3dp_pla_shims_4_loads_reference")

## Problem info

NOTE: Either define the problem info here or load it from an optimization file.

In [ ]:
# NOTE: Units are mm, N, s

# Retrieve design info from optimization data
optimization_filename = f"opt_integrated_with_angle_30_and_length_5_constraints_quads_15x15_amplitude_4.50_loading_rate_2.0_initial_angle_20.0_clamping_corners_True_damping_scaling_1.0"
optimization = OptimizationProblem.from_dict(
    load_data(
        f"{data_folder}/{optimization_filename}.pkl",
    )
)
problem = optimization.forward_problem
# Set up the forward problem to get the geometry
problem.setup()
geometry = problem.cloaked_geometry

# Select the initial design as it is also the reference rotating square design
design_values = optimization.design_values[0]

Setup intact case dynamics
Setup the solver of the shielding cloak dynamics


## Rename video files

In [4]:
# # NOTE: Run this cell to rename the video files in human readable format!

# # Grab the video files
# # Assumes filenames are in a known order
# video_paths = sorted(
#     list(Path(f"{data_folder}/videos").glob("*.mp4"))
# )

# # Define the experimental parameters used to generate the videos (in the same order as the video files)
# loading_angles = [3*jnp.pi/4]*6 + [jnp.pi/2]*6 + [jnp.pi/4]*6 + [0.]*6
# voltages = [65, 70, 75]*8
# frequencies = [2.]*len(video_paths)
# trial_ids = [1, 1, 1, 2, 2, 2]*4

# assert len(voltages) == len(frequencies) == len(
#     trial_ids) == len(loading_angles), "Lengths of experimental parameters do not match the number of video files"

# # Rename the files
# for path, loading_angle, voltage, frequency, trial_id in zip(video_paths, loading_angles, voltages, frequencies, trial_ids):
#     new_path = path.parent / \
#         f"reference_shielding_loading_angle_{loading_angle*180/jnp.pi:.1f}_voltage_{voltage:.0f}mV_frequency_{frequency:.0f}Hz_trial_{trial_id:02d}_{path.stem}{path.suffix}"
#     new_path.parent.mkdir(parents=True, exist_ok=True)
#     path.replace(new_path)

## Videos to process


In [5]:
# Experimental videos to be processed
video_paths = sorted(list(Path(f"{data_folder}/videos").glob("*.mp4")))

## Tracking


### Tracking helper functions



In [9]:
# Define the tracked geometry i.e the reference geometry without clamped blocks
reference_centroids = geometry.block_centroids(
    *problem.cloak_to_all_shifts(design_values)
)[problem.moving_blocks_ids]
reference_shapes = geometry.centroid_node_vectors(
    *problem.cloak_to_all_shifts(design_values)
)[problem.moving_blocks_ids]


def morphological_transformation(thresh):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    transformed = cv2.morphologyEx(
        thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    transformed = cv2.morphologyEx(
        transformed, cv2.MORPH_CLOSE, kernel, iterations=1)
    # kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    # transformed = cv2.morphologyEx(transformed, cv2.MORPH_ERODE, kernel, iterations=1)
    return transformed


def add_clamped_blocks_data(tracked_data: SolutionData) -> SolutionData:
    """Add the data for the clamped blocks to the tracked data (for easier comparison with simulations)."""
    block_centroids = geometry.block_centroids(
        *problem.cloak_to_all_shifts(design_values))
    centroid_node_vectors = geometry.centroid_node_vectors(
        *problem.cloak_to_all_shifts(design_values))
    fields = jnp.zeros((len(tracked_data.timepoints),
                       2, len(block_centroids), 3))

    block_centroids = block_centroids.at[problem.moving_blocks_ids].set(
        tracked_data.block_centroids)
    centroid_node_vectors = centroid_node_vectors.at[problem.moving_blocks_ids].set(
        tracked_data.centroid_node_vectors
    )
    fields = fields.at[:, :, problem.moving_blocks_ids].set(
        tracked_data.fields)

    return SolutionData(
        block_centroids=block_centroids,
        centroid_node_vectors=centroid_node_vectors,
        bond_connectivity=geometry.bond_connectivity(),
        timepoints=tracked_data.timepoints,
        fields=fields,
    )


# Marker placement parameters
marker_placement_params = dict(
    calib_xy=(14*15./543, 14*15./543),
    ROI_X=(64, 714),
    ROI_Y=(251, 901),
    blur_size=1,
    threshold=30,
    adaptive_thresholding=True,
    adaptive_thresholding_block=1001,
    morphological_transformation=morphological_transformation,
    block_area=(30, 2000),
    reference_centroids=reference_centroids,
    reference_shapes=reference_shapes,
    aspect_ratio_threshold=0.05,
    # Place markers a little bit closer to the centroid to have better features for cross-correlation.
    markers_scaled_position=0.95,
)
# Cross-correlation parameters
xcorr_params = dict(
    marker_template_size=21,  # px
    search_window_size=31,  # px
    upscaling_factor=5,
    template_update_rate=0,
)


def show_reference_frame(video_path: Union[str, Path], frame: int = 0):
    mark_reference_frame(
        video_path=str(video_path),
        **marker_placement_params,
        frame=frame,
        show=True,
    )


def track_video(video_path: Union[str, Path], plot_centroids: bool = False, print_frame_number: bool = False, show_tracked_frame: bool = False):
    tracked_data = tracking(
        video_path=str(video_path),
        start_end_video=(0, -1),
        framerate=300,
        step_size=1,
        **marker_placement_params,
        **xcorr_params,
        print_frame_number=print_frame_number,
        show_tracked_frame=show_tracked_frame,
    )
    # Add the data for the clamped blocks
    tracked_data = add_clamped_blocks_data(tracked_data)
    # Shift timepoints to start at 0
    tracked_data = tracked_data._replace(
        timepoints=tracked_data.timepoints - tracked_data.timepoints[0])
    # Smooth the data with a Savitzky-Golay filter
    # tracked_data._replace(fields=smooth_fields_SG(tracked_data.fields, window_length=11, polyorder=3))

    if plot_centroids:
        # Plot centroids
        plt.figure(figsize=(8, 8*(geometry.n2_blocks+1) /
                   geometry.n1_blocks), constrained_layout=True)
        plt.title(r"Centroids - frame \#0")
        plt.scatter(*reference_centroids.T, label="Reference")
        plt.scatter(
            *tracked_data.block_centroids.T, label="Tracked")
        for i, pt in enumerate(tracked_data.block_centroids):
            plt.text(*pt, f"{i}")
        plt.axis("equal")
        plt.legend(bbox_to_anchor=(0.5, -0.08), loc='center', ncol=3)

    return tracked_data

### Process videos


#### Place markers

In [ ]:
show_reference_frame(video_paths[0], frame=0)


#### Run tracking and save the tracking data

In [ ]:
# # Track each video and save the tracked data
# for path in video_paths:
#     tracked_data = track_video(path)
#     save_data(f"{data_folder}/dynamic-data/tracking_exp/{path.stem}.pkl", tracked_data._asdict())


## Load tracking data


In [ ]:
# Find all data files in the dynamic-data folder
tracked_data_paths = sorted(list(Path(
    f"{data_folder}/dynamic-data/tracking_exp/").glob("*.pkl")), key=lambda p: p.stem[-2:])

# Load the tracked data as a dataframe
tracked_data = pd.DataFrame({
    "label": [p.stem for p in tracked_data_paths],
    "data": [SolutionData(**load_data(p)) for p in tracked_data_paths],
})

# Smooth tracking data
tracked_data.data = tracked_data.apply(
    lambda row: row.data._replace(fields=smooth_fields_SG(
        row.data.fields, window_length=[[10, 10, 25], [10, 10, 25]], polyorder=2)),
    axis=1,
)
tracked_data


## Animation of tracked experiments


In [ ]:
# Animation tracked experiments
xlim, ylim = problem.geometry.get_xy_limits(
    *problem.cloak_to_all_shifts(design_values)
) + 0.5*problem.geometry.spacing * jnp.array([-1, 1])

for row in tracked_data.itertuples():
    generate_animation(
        row.data,
        frame_range=jnp.arange(0, row.data.timepoints.shape[0], 2),
        field="theta_abs",
        out_filename=f"{out_folder}/tracking_exp/{row.label}",
        xlim=xlim,
        ylim=ylim,
        figsize=(6, 5),
        fps=30,
        dpi=300,
        cmap="inferno",
    )